# MapBiomas Argentina — Fuego Colección 1
## Paso 10 · validación — sorteo de 40.000 puntos por estrato (S1/S2/S3)

### Qué hace este notebook

Para un año-fuego dado, sortea 40.000 píxeles de cada estrato (S1 = quemado, S2 = borde/evidencia,
S3 = resto) sobre el raster de estratos ya aterrizado (`sampling_strata_fy<AÑO>`), y exporta cada
estrato como un CSV a tu Google Drive (carpeta `mapbiomas_fire_validation_10`). Son 3 tareas por
año-fuego — este paso solo **lee** assets de GEE, nunca escribe ni modifica ninguno.

### Antes de correr algo, verificá esto

1. **El año-fuego ya tiene que tener el raster de estratos aterrizado** (paso 01 de la
   validación). Si no estás seguro, avisale a Ramón antes de lanzar — la Celda 4 avisa si falta
   y no manda nada.
2. **Tu cuenta de Google necesita permiso de LECTURA** (no de escritura) sobre estos cuatro
   assets de `mapbiomas-argentina`:
   - `projects/mapbiomas-argentina/assets/FIRE/VALIDATION/sampling_strata`
   - `projects/mapbiomas-argentina/assets/ANCILLARY_DATA/VECTOR/ARG/ARG-Political_Level_1-Pais`
   - `projects/mapbiomas-argentina/assets/LAND-COVER/COLLECTION-2/INTEGRATION/mapbiomas_argentina_collection1_integration_v8_buffer`
   - `projects/mapbiomas-argentina/assets/ANCILLARY_DATA/RASTER/ARG/ARG-Regiones-MapBiomas-buffer2km`

   Si te falta alguno, pedile a Ramón que te agregue como lector desde el Code Editor (botón
   *Share* en cada asset).
3. **Proyecto de cómputo** (Celda 2): podés dejar tu propio proyecto (`mapbiomas-fire-485203`) —
   así no competís por recursos con el resto de `mapbiomas-argentina`, bastante saturado estos
   días por tareas de otros equipos. No hace falta "agregar" `mapbiomas-argentina` a tu cuenta
   para esto: el proyecto de cómputo (a qué cuota se factura) y el proyecto donde viven los
   assets (`mapbiomas-argentina`, fijo en el código) son cosas independientes.

### ⚠️ Importante — no hay protección contra relanzar un estrato ya terminado

A diferencia de otras exportaciones del repo, este paso NO se fija si un estrato ya se exportó
con éxito — solo evita relanzar uno que esté `PENDING`/`RUNNING`. Si volvés a correr la Celda 4
para un año/estrato que ya terminó bien, vas a mandar un sorteo duplicado. Corré la Celda 4
**una sola vez** por año, salvo que un estrato haya fallado explícitamente.

### Al terminar

Las tres tareas exportan a **tu** Google Drive (carpeta `mapbiomas_fire_validation_10`), no al de
Ramón. Cuando terminen, compartile esa carpeta (o mandale los 3 CSV) para que pueda congelar la
lista (`--freeze`).


In [ ]:
# Celda 1 — Configuración. Ejecutar una vez por sesión (también al volver después de que Colab caduque).
!pip install -q -U earthengine-api
!git clone -b ramon/step10-validation https://github.com/barberaivan/mapbiomas-argentina-fire.git 2>/dev/null || (cd mapbiomas-argentina-fire && git pull -q)

import importlib
import sys
sys.path.insert(0, '/content/mapbiomas-argentina-fire/collection-01/validation')

# El código de este paso vive en validation/02_sample_pool.py. Como el nombre del archivo
# empieza con un dígito, Python no lo puede importar con `import 02_sample_pool`; se carga con
# import_module, mismo truco que ya usa el propio script para leer 01_strata_export.py.
step02 = importlib.import_module('02_sample_pool')
print('repositorio clonado, funciones importadas — listo')

In [ ]:
# Celda 2 — Autenticación. Iniciá sesión con la cuenta de Google que tiene lectura sobre los
# assets de mapbiomas-argentina listados arriba. Al volver después de que Colab caduque hay que
# autenticarse de nuevo.
# GEE_PROJECT es el proyecto de CÓMPUTO (a qué cuota se factura) — independiente de dónde viven
# los assets, que siempre son de mapbiomas-argentina. Usar tu propio proyecto evita competir por
# recursos con el resto de mapbiomas-argentina, saturado estos días por tareas de otros equipos.
GEE_PROJECT = 'mapbiomas-fire-485203'   # o 'mapbiomas-argentina' si preferís / no tenés otro

import ee
ee.Authenticate()
step02.initialize(GEE_PROJECT)
print('inicializado — proyecto de cómputo:', GEE_PROJECT)

In [ ]:
# Celda 3 — ✏️ Poné acá el año-fuego que vas a sortear (tiene que tener ya el raster de
# estratos aterrizado — paso 01 de la validación).
YEAR = 2013

In [ ]:
# Celda 4 — Chequea que el raster de estratos exista y manda las 3 tareas (S1/S2/S3).
# ⚠️ No la vuelvas a correr para un año/estrato que ya terminó bien — no hay chequeo de
# "ya exportado", solo de tareas en curso (ver más arriba).
step02.check(YEAR)
step02.launch(YEAR)
print('listo — las tareas (si no estaban ya en curso) quedaron corriendo en GEE; podés cerrar esta pestaña.')

## Notas y resolución de problemas

- **Monitoreo:** las tareas enviadas aparecen en la pestaña **Tasks** del [Code Editor](https://code.earthengine.google.com/) (misma cuenta de Google que usaste en la Celda 2). También podés volver a correr `step02.check(YEAR)` (sin `launch`) para ver si el asset de estratos está — eso no manda nada a GEE.
- **No hay retomar/reenviar parcial:** a diferencia de otros pasos de este repo, acá son 3 tareas fijas por año (una por estrato) y el único chequeo antes de mandarlas es que no estén ya `PENDING`/`RUNNING`. Si una falló, podés volver a correr la Celda 4 sin problema — solo evitá relanzar un estrato que ya terminó bien.
- **Dónde caen los CSV:** en tu Google Drive, carpeta `mapbiomas_fire_validation_10`, con nombre `val10_sample_fy<AÑO>_s<1|2|3>.csv`. Compartile esa carpeta (o mandale los 3 archivos) a Ramón cuando terminen — el paso siguiente (`--freeze`) los necesita.
- **Permisos (causa más común de error):** tu cuenta de Google tiene que poder *leer* los cuatro assets de `mapbiomas-argentina` listados arriba. Si la Celda 4 falla de inmediato con un error de permisos, es eso — pedile a Ramón que te agregue como lector desde el Code Editor (*Share* en cada asset).